# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad_Debug/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow_ConvStudy", myBatch);

Project name is set to 'KovasznayFlow_ConvStudy'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\KovasznayFlow_ConvStudy'.


## Grid generation

In [ ]:
int[] Resolutions = new int[] { 1, 2, 4, 8, 16, 32, 64, 128, 256 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"KovasznayFlow_{Res}x{2*Res}_DongBC";

    // IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    IGridInfo cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(-0.5, -0.1, Res + 1);
        double[] yNodes = GenericBlas.Linspace(-0.5, 0.5, (2 * Res) + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicY:true);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if((X.x + 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.Velocity_Inlet.ToString();
            if((X.x + 0.1).Abs() <= 1e-8)
                ret = IncompressibleBcType.Dong_OutFlow.ToString();
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Grid Edge Tags changed.
An equivalent grid (382b2389-a538-4fbd-9a9d-b1e4fbe8b6a5) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (7e4dac7c-5969-4697-a759-e1816f68d85a) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (6d97b03a-3bb3-480e-8b6d-f7e2360ed3e0) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (8bd9c2f6-ef16-4a0e-9fa6-bf7c75fb015f) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (532b5577-f08c-45e8-ad22-8041bd592802) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (2da07b05-5205-4c65-bac5-632096eb2636) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (9bf54200-be7d-4d72-89b8-4c116a3e79f7) is already present in the data

## Initial Values

In [ ]:
Formula KovasznayFlow_u = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velX = 1.0 - (Math.Exp(lambda * X[0]) * Math.Cos(2.0 * Math.PI * X[1]));" +
    "return velX; } "
);

In [ ]:
Formula KovasznayFlow_v = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velY = (lambda/(2.0 * Math.PI)) * (Math.Exp(lambda * X[0]) * Math.Sin(2.0 * Math.PI * X[1]));" +
    "return velY; } "
);

In [ ]:
Formula KovasznayFlow_p = new Formula(
    "Pres",
    false,
    "double Pres(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double Pres = 0.5 * (1.0 - Math.Exp(2.0 * lambda * X[0]));" +
    "return Pres; } "
);

## Setting up test solver (with additional terms for manufactured solution)

In [ ]:
using BoSSS.Foundation.XDG.OperatorFactory;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.XNSECommon; 

In [ ]:
public class DongBoundaryCondition_KovasznayFlow : IEdgeForm, ISupportsJacobianComponent, ISpeciesFilter {


    public DongBoundaryCondition_KovasznayFlow(string spcName, int d, double rho, double mu, IncompressibleMultiphaseBoundaryCondMap BcMap) {
        ValidSpecies = spcName;
        m_d = d;
        m_rho = rho;
        m_mu = mu;

        m_BcMap = BcMap;
    }

    int m_d;
    double m_rho;
    double m_mu;

    IncompressibleMultiphaseBoundaryCondMap m_BcMap;


    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] _uA, double[,] _Grad_uA, double _vA, double[] _Grad_vA) {
        double Flx_InCell = 0.0;

        if (m_BcMap.EdgeTag2Type[inp.EdgeTag] == IncompressibleBcType.Dong_OutFlow) {

            double nu = m_mu / m_rho;
            double lambda = (1.0 / (2.0 * nu)) - Math.Sqrt(1.0 / (4.0 * nu.Pow2()) + 4.0 * Math.PI.Pow2());

            double x = inp.X[0];
            double y = inp.X[1];

            double p = 0.5 * (1.0 - Math.Exp(2 * lambda * x));

            double u = 1.0 - Math.Exp(lambda * x) * Math.Cos(2.0 * Math.PI * y);
            double v = (lambda / (2.0 * Math.PI)) * Math.Exp(lambda * x) * Math.Sin(2.0 * Math.PI * y);

            //Matrix(2, 2, [[-lambda*exp(lambda*x)*cos(2*pi*y), 2*exp(lambda*x)*pi*sin(2*pi*y)], [1/2*lambda^2*exp(lambda*x)*sin(2*pi*y)/pi, lambda*exp(lambda*x)*cos(2*pi*y)]])

            double u_x = -lambda * Math.Exp(lambda * x ) * Math.Cos(2.0 * Math.PI * y);
            double u_y = 2.0 * Math.Exp(lambda * x) * Math.PI * Math.Sin(2.0 * Math.PI * y);

            double v_x = 0.5 * lambda.Pow2() * Math.Exp(lambda * x) * Math.Sin(2.0 * Math.PI * y) / Math.PI;
            double v_y = lambda * Math.Exp(lambda * x) * Math.Cos(2.0 * Math.PI * y);

            double[,] GradU = new double[2, 2] { {u_x, u_y}, {v_x, v_y} };
    
            // pressure gradient
            Flx_InCell += - p * inp.Normal[m_d];
    
            for (int d = 0; d < 2; d++) {
                // viscous terms 
                Flx_InCell += m_mu * ( GradU[m_d, d] + GradU[d, m_d] ) * inp.Normal[d];
            }

            // Dong term
            double U0 = 1.0;
            double delta = 1.0 / 20.0;
            double ndotu = (u * inp.Normal[0] + v * inp.Normal[1]);
            double Sout = 0.5 * (1.0 - Math.Tanh(ndotu / (U0 * delta)));
            double uAbs2 = u.Pow2() + v.Pow2();

            Flx_InCell += 0.5 * uAbs2 * Sout * inp.Normal[m_d];

        }

        return - Flx_InCell * _vA;
    }


    public double InnerEdgeForm(ref CommonParams inp, double[] _uA, double[] _uB, double[,] _Grad_uA, double[,] _Grad_uB, double _vA, double _vB, double[] _Grad_vA, double[] _Grad_vB) {
        return 0.0;
    }

    public IList<string> ArgumentOrdering { get { return new string[0]; }  }

    public IList<string> ParameterOrdering { get { return new string[0]; } }

        
    virtual public TermActivationFlags BoundaryEdgeTerms {
        get { return TermActivationFlags.V; }
    }

    virtual public TermActivationFlags InnerEdgeTerms {
        get { return TermActivationFlags.None; }
    }

    public string ValidSpecies {
        get;
        private set;
    }

    virtual public IEquationComponent[] GetJacobianComponents(int SpatialDimension) {
        return new IEquationComponent[] { this };
    }


}

In [ ]:
public class ManufacturedSolutionComp : BulkEquation {

    public ManufacturedSolutionComp(int d, IncompressibleMultiphaseBoundaryCondMap BcMap, XNSE_OperatorConfiguration config, string spcName) {
        m_CodomainName = EquationNames.MomentumEquationComponent(d);
        m_SeciesName = spcName;

        PhysicalParameters physParams = config.getPhysParams;

        double rhoSpc, muSpc;
        switch (spcName) {
            case "A": { rhoSpc = physParams.rho_A; muSpc = physParams.mu_A; break; }
            case "B": { rhoSpc = physParams.rho_B; muSpc = physParams.mu_B; break; }
            default: throw new ArgumentException("Unknown species.");
        }
        m_rho = rhoSpc;

        AddComponent(new DongBoundaryCondition_KovasznayFlow(spcName, d, rhoSpc, muSpc, BcMap));

    }


    string m_CodomainName;

    string m_SeciesName;

    double m_rho;

    public override string CodomainName { get { return m_CodomainName; } }

    public override string SpeciesName { get { return m_SeciesName; } }

    public override double MassScale { get { return m_rho; } }


}

In [ ]:
public class XNSE_Test : XNSE_Test<XNSE_Test_Control> {

    // ===========
    //  Main file
    // ===========
    static void Main(string[] args) {
        XNSE_Test._Main(args, false, delegate () {
            var p = new XNSE_Test();
            return p;
        });
    }
}

public class XNSE_Test<T> : XNSE<T> where T : XNSE_Test_Control, new() {

    protected override void DefineSystem(int D, OperatorFactory opFactory, LevelSetUpdater lsUpdater) {
        base.DefineSystem(D, opFactory, lsUpdater);

        XNSE_OperatorConfiguration config = new XNSE_OperatorConfiguration(this.Control);

        if (D != 2)
            throw new NotSupportedException("Test setup only for 2D case");

        for (int d = 0; d < D; ++d) {
            opFactory.AddEquation(new ManufacturedSolutionComp(d, boundaryMap, config, "A"));
            opFactory.AddEquation(new ManufacturedSolutionComp(d, boundaryMap, config, "B"));
        }
    }

}

public class XNSE_Test_Control : XNSE_Control {

    public override Type GetSolverType() {
        return typeof(XNSE_Test<XNSE_Test_Control>);
    }
}

## Setting up control 

In [ ]:
int[] polyDeg = { 2, 3 };

In [ ]:
List<XNSE_Test_Control> Controls = new List<XNSE_Test_Control>();
Controls.Clear();

In [ ]:
foreach (int k in polyDeg) {    // loop over poylnomial degrees
foreach (var grd in Grids) {    // loop over grids
//for (int res = 0; res < Resolutions.Length; res++) {
    var C = new XNSE_Test_Control();

    C.SetDGdegree(k);

    // physical parameters
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.mu_A = 1.0 / 40.0;

    C.PhysicalParameters.IncludeConvection = true;

    C.SetGrid(grd);
    int J = grd.NumberOfCells * ((((k + 1)*(k + 1)) + (k + 1)) / 2);

    // initial conditions
    C.AddInitialValue("VelocityX#A", KovasznayFlow_u);
    C.AddInitialValue("VelocityY#A", KovasznayFlow_v);
    C.AddInitialValue("Pressure#A", KovasznayFlow_p);

    // boundary conditions
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", KovasznayFlow_u);
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityY#A", KovasznayFlow_v);

    //C.AddBoundaryValue(IncompressibleBcType.Dong_OutFlow.ToString(), "VelocityX#A", KovasznayFlow_u);
    //C.AddBoundaryValue(IncompressibleBcType.Dong_OutFlow.ToString(), "VelocityY#A", KovasznayFlow_v);

    // C.SkipSolveAndEvaluateResidual = true;
    //C.CutCellQuadratureType = XQuadFactoryHelper.MomentFittingVariants.OneStepGaussAndStokes;

    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
    C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.Timestepper_LevelSetHandling = LevelSetHandling.None;
    C.Option_LevelSetEvolution = LevelSetEvolution.None;

    // C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    // C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    // C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    // C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
    // C.dtFixed = 2.5e-5;
    // C.NoOfTimesteps = 4; 

    C.SessionName = $"DongBC_KovasznayFlow_ConvStudy_p{k}_J{J}_Picard";
    
    Controls.Add(C);
}
}

In [ ]:
Controls.Select(ctrl => ctrl.SessionName)

index,value
0,DongBC_KovasznayFlow_ConvStudy_p2_J12_Picard
1,DongBC_KovasznayFlow_ConvStudy_p2_J48_Picard
2,DongBC_KovasznayFlow_ConvStudy_p2_J192_Picard
3,DongBC_KovasznayFlow_ConvStudy_p2_J768_Picard
4,DongBC_KovasznayFlow_ConvStudy_p2_J3072_Picard
5,DongBC_KovasznayFlow_ConvStudy_p2_J12288_Picard
6,DongBC_KovasznayFlow_ConvStudy_p2_J49152_Picard
7,DongBC_KovasznayFlow_ConvStudy_p2_J196608_Picard
8,DongBC_KovasznayFlow_ConvStudy_p2_J786432_Picard
9,DongBC_KovasznayFlow_ConvStudy_p3_J20_Picard


In [ ]:
int[] numProcs = { 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 2, 8 };
numProcs.Length

18

In [ ]:
myBatch.Name

FDY-WindowsHPC

In [ ]:
//Controls.Pick(0).Run();
for (int i = 0; i < Controls.Count; i++) {
    var oneJob              = Controls[i].CreateJob();
    
    oneJob.NumberOfMPIProcs = numProcs[i];

    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    oneJob.Activate(myBatch); 
}

Error: System.ApplicationException: internal error in assembly collection.
   at ilPSP.AssemblyDeploymentExtensions.GetAllDependentAssembliesRecursive(Assembly a, HashSet`1 assiList, String SearchPath) in D:\BoSSS-experimental\public\src\ilPSP\layer_1.2-ilPSP\ilPSP\AssemblyDeploymentExtensions.cs:line 29
   at ilPSP.AssemblyDeploymentExtensions.GetAllDependentAssembliesRecursive(Assembly a, HashSet`1 assiList, String SearchPath) in D:\BoSSS-experimental\public\src\ilPSP\layer_1.2-ilPSP\ilPSP\AssemblyDeploymentExtensions.cs:line 47
   at ilPSP.AssemblyDeploymentExtensions.GetAllDependentAssembliesRecursive(Assembly a, HashSet`1 assiList, String SearchPath) in D:\BoSSS-experimental\public\src\ilPSP\layer_1.2-ilPSP\ilPSP\AssemblyDeploymentExtensions.cs:line 47
   at ilPSP.AssemblyDeploymentExtensions.GetAllDependentAssembliesRecursive(Assembly a, HashSet`1 assiList, String SearchPath) in D:\BoSSS-experimental\public\src\ilPSP\layer_1.2-ilPSP\ilPSP\AssemblyDeploymentExtensions.cs:line 

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: empty-project-name	DongBC_KovasznayFlow_testSolve	03/28/2025 14:57:34	d4423545...
#1: empty-project-name	DongBC_KovasznayFlow_testSolve	03/28/2025 14:27:32	f5662681...
#2: empty-project-name	DongBC_KovasznayFlow_testSolve*	03/28/2025 14:21:11	4bdb7b89...
#3: empty-project-name	DongBC_KovasznayFlow_testSolve*	03/28/2025 14:20:05	e2bc0a3c...
#4: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 14:11:56	e855bf94...
#5: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 14:10:33	111d626d...
#6: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 14:02:11	3f1fdbe7...
#7: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 13:44:08	f1634926...
#8: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 13:22:32	a98a0ce4...
#9: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 12:39:20	21c6044d...
#10: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 12:38:34	b5348634...
#11: empty-project-name	DongBC_KovasznayFlow_test	03/28/2025 12:30:12	956a597a...
#12:

In [ ]:
var sess = sessions.Pick(0);
sess

empty-project-name	DongBC_KovasznayFlow_testSolve	03/28/2025 14:57:34	d4423545...

In [ ]:
var ts = sess.Timesteps.Last();
ts

 { Time-step: 1; Physical time: 1.7976931348623158E+304s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, MPIrank, CutCells; Name:  }

In [ ]:
ts.GetField("VelocityX").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
    KovasznayFlow_u.EvaluateV(nodes, 0.0, results);
})

0.0008146637459666468

In [ ]:
ts.GetField("VelocityY").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
    KovasznayFlow_v.EvaluateV(nodes, 0.0, results);
})

0.00019771265024459564

In [ ]:
ts.GetField("Pressure").L2Error((ScalarFunction)delegate (MultidimensionalArray nodes, MultidimensionalArray results) {
    KovasznayFlow_p.EvaluateV(nodes, 0.0, results);
})

0.009182100655382747